In 20260127_pairwise_to_homeo_weights.ipynb, we found the best weight for each pairwise comparison in respect to homeostatic need. We found a range of weights that can be used for each objective term without penalizing or sacrificing the homeostatic need. Now we want to find the best combination of weights for all terms together. In this notebook, I will be using the epsilon constraint method to find the best combination of weights for all terms together. We have a vague idea of the order of importance of the terms: we value homeostatic objective the most, then kinetics, then efficiency, secretion, and diversity are kind of equal. Secondly, through pareto_exploration.py, we found several sets of weight combinations that can be used without sacrificing the homeostatic need, as an example: lam_sec = 4.07e-10, lam_eff = 8.56e-7, lam_kin = 1.60e-3, lam_div=2.57e-7. We will use these as a prior for our epsilon constraint method.

We will fix the homeostatic term to be within a certain range of the optimal value, and then we will vary the weights of the other terms to find the best combination of weights that maximizes the other terms while still satisfying the homeostatic constraint.

In [20]:
import altair
import pandas as pd
import os, dill
import numpy as np
import cvxpy as cp

from ecoli.processes.metabolism_redux_classic import NetworkFlowModel, FlowResult

os.chdir(os.path.expanduser('~/dev/vEcoli'))

%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [21]:
def load_sim(
        time_num: int,
        date: str,
        experiment_name: str,
        condition: str,
        experiment_type: str,
):
    # --- Load Sim ---
    time = str(time_num)
    entry = f'{experiment_name}_{time}_{date}'
    folder_path = f'out/{experiment_type}/{condition}/{entry}/'

    output = np.load(folder_path + '0_output.npy',allow_pickle='TRUE').item()
    output = output['agents']['0']
    fba = output['listeners']['fba_results']
    bulk = pd.DataFrame(output['bulk'])
    f = open(folder_path + 'agent_steps.pkl', 'rb')
    agent = dill.load(f)
    f.close()

    metabolism = agent['ecoli-metabolism-redux-classic']

    return fba, bulk, metabolism, output

In [22]:
# import sim
time_num = 600
# date = '2026-01-22'
date = '2026-04-06'
experiment_name = 'homeostatic_only'
condition = 'basal'
experiment_type = 'objective_weight'

fba, bulk, metabolism, output = load_sim(time_num, date, experiment_name, condition, experiment_type)

In [23]:
stoichiometry = metabolism.stoichiometry.copy()
reaction_names = metabolism.reaction_names
metabolites = metabolism.metabolite_names.copy()
counts_to_molar = output['listeners']['enzyme_kinetics']['counts_to_molar']

homeostatic_only_term = np.array(fba['homeostatic_term'].copy())/counts_to_molar # covert to counts
homeostatic_only_baseline = np.max(homeostatic_only_term) # in counts

homeostatic_dm_targets =  pd.DataFrame(fba['target_homeostatic_dmdt'], columns=metabolism.homeostatic_metabolites).mul(counts_to_molar, axis=0).max(axis=0)
homeostatic_metabolite_counts = pd.DataFrame(fba['homeostatic_metabolite_counts'], columns=metabolism.homeostatic_metabolites).mul(counts_to_molar, axis=0).mean(axis=0)
maintenance = pd.DataFrame(fba["maintenance_target"][1:], columns=['maintenance_reaction']).mul(counts_to_molar[1:], axis=0).mean(axis=0)
kinetic = pd.DataFrame(fba["target_kinetic_fluxes"], columns=metabolism.kinetic_constraint_reactions).mul(counts_to_molar, axis=0).mean(axis=0)

In [25]:
import ecoli.processes.metabolism_redux_classic as metabolism_redux_classic


<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 9475 stored elements and shape (2829, 9475)>

In [7]:
homeostatic_dm_targets =  pd.DataFrame(fba['target_homeostatic_dmdt'], columns=metabolism.homeostatic_metabolites).mul(counts_to_molar, axis=0).iloc[1]
homeostatic_metabolite_counts = pd.DataFrame(fba['homeostatic_metabolite_counts'], columns=metabolism.homeostatic_metabolites).mul(counts_to_molar, axis=0).iloc[1]
maintenance = pd.DataFrame(fba["maintenance_target"][1:], columns=['maintenance_reaction']).mul(counts_to_molar[1:], axis=0).iloc[1]
kinetic = pd.DataFrame(fba["target_kinetic_fluxes"], columns=metabolism.kinetic_constraint_reactions).mul(counts_to_molar, axis=0).iloc[1]

os.chdir(os.path.expanduser('~/dev/vEcoli/notebooks/Heena notebooks/Metabolism_New Genes'))
from pareto_exploration import run

df = run(
    stoichiometry=stoichiometry,
    metabolites=metabolites,
    reaction_names=reaction_names,
    metabolism=metabolism,
    homeostatic_metabolite_counts=homeostatic_metabolite_counts,
    homeostatic_dm_targets=homeostatic_dm_targets,
    kinetic=kinetic,
    maintenance=maintenance,
    counts_to_molar=counts_to_molar,
    n_samples=1000,
    n_jobs=5,
)

Sampling 1000 weight combinations (log-uniform in feasible ranges)...
Solving 1000 problems (5 parallel job(s))...



  2%|▏         | 15/1000 [00:03<03:09,  5.19it/s]/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(

  4%|▎         | 35/1000 [00:08<03:43,  4.33it/s]

KeyboardInterrupt: 

In [7]:
df

lambda_sec,lambda_eff,lambda_kin,lambda_div,obj_total,obj_homeo,obj_kin,obj_eff,obj_sec,obj_div
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.000353,0.000006,0.003282,0.003087,0.069609,0.019302,12.179734,301.9015,24.507251,0.003524
0.000075,0.000071,0.000512,0.00861,0.037559,0.000736,31.263228,271.95076,18.360109,0.00344
0.000521,0.000019,0.008738,0.000448,0.144049,0.01998,12.109638,294.656254,24.05123,0.004107
0.000248,0.000044,0.006113,0.000551,0.112809,0.020199,12.109638,285.32349,24.055077,0.004036
0.000015,0.000008,0.003604,0.000869,0.065901,0.019302,12.179708,300.374657,25.082489,0.003564
0.000894,0.000003,0.000245,0.000239,0.018196,3.4390e-12,42.557321,247.575825,7.9001,0.004273
0.000333,0.000013,0.000858,0.000182,0.035973,4.1638e-12,24.609489,359.523478,30.753139,0.004282
0.000373,0.000001,0.000122,0.000894,0.008492,4.9724e-12,42.558155,247.580988,7.900389,0.003477
0.000018,0.000045,0.000204,0.000284,0.019276,0.000127,36.899754,252.250715,12.949534,0.004074


In [8]:
import altair as alt
terms = [
    ("obj_sec", "lambda_sec", "Secretion"),
    ("obj_eff", "lambda_eff", "Efficiency"),
    ("obj_kin", "lambda_kin", "Kinetic"),
    ("obj_div", "lambda_div", "Diversity"),
]

# --- Make Scatter Plots ---
charts = []
interval = alt.selection_interval()
for obj_col, lam_col, title in terms:
    chart = (
        alt.Chart(df)
        .mark_circle(size=40, opacity=0.6)
        .encode(
            y=alt.Y("obj_homeo:Q", title="Homeostatic Objective"),
            x=alt.X(f"{obj_col}:Q", title=f"{title} Objective"),
            color=alt.condition(
                interval,
                alt.Color(
                    f"{lam_col}:Q",
                    scale=alt.Scale(scheme="viridis"),
                    title=f"λ_{title[:3].lower()}",
                ),
                alt.value("lightgray"),
            ),
            tooltip=[
                alt.Tooltip("obj_homeo:Q", format=".4e"),
                alt.Tooltip(f"{obj_col}:Q", format=".4e"),
                alt.Tooltip(
                    f"{lam_col}:Q", format=".2e", title=f"λ_{title[:3].lower()}"
                ),
            ],
        )
        .properties(title=f"Homeostatic vs {title}", width=280, height=250)
    ).add_selection(interval)
    charts.append(chart)

/var/folders/s9/gn2yly0s7rzgcc2xyvt7nsxm0000gr/T/ipykernel_35267/1569619653.py:37: AltairDeprecationWarning:


Deprecated since `altair=5.0.0`. Use add_params instead.



In [9]:
combined_scatter = (
        ((charts[0] | charts[1]) & (charts[2] | charts[3]))
        .properties(title="Pairwise Pareto: Homeostatic vs Secondary Objectives")
    )

In [15]:
ranked_text = (
    alt.Chart(df)
    .mark_text(align='right')
    .encode(y=alt.Y('rank:O', axis=None))
    .transform_filter(interval)
    .transform_calculate(
        lambda_norm="sqrt(datum.lambda_sec * datum.lambda_sec + "
                    "datum.lambda_eff * datum.lambda_eff + "
                    "datum.lambda_kin * datum.lambda_kin + "
                    "datum.lambda_div * datum.lambda_div)"
    )
    .transform_window(
        rank='rank()',
        sort=[alt.SortField('lambda_norm', order='descending')],
    )
    .transform_filter(alt.datum.rank <= 15)
    .properties(height=300)
)

In [16]:
# Data Tables
lambda_sec = ranked_text.encode(text=alt.Text('lambda_sec:Q', format=".2e")).properties(
    title=alt.Title(text='λ_sec', align='right')
)
lambda_eff = ranked_text.encode(text=alt.Text('lambda_eff:Q', format=".2e")).properties(
    title=alt.Title(text='λ_eff', align='right')
)
lambda_kin = ranked_text.encode(text=alt.Text('lambda_kin:Q', format=".2e")).properties(
    title=alt.Title(text='λ_kin', align='right')
)
lambda_div = ranked_text.encode(text=alt.Text('lambda_div:Q', format=".2e")).properties(
    title=alt.Title(text='λ_div', align='right')
)
obj_homeo = ranked_text.encode(text=alt.Text('obj_homeo:Q', format=".3e")).properties(
    title=alt.Title(text='Homeostatic Objective', align='right')
)
obj_kinetic = ranked_text.encode(text=alt.Text('obj_kin:Q', format=".3f")).properties(
    title=alt.Title(text='Unweighted Kinetic Objective', align='right')
)
text = alt.hconcat(lambda_sec, lambda_eff, lambda_kin, lambda_div, obj_homeo, obj_kinetic)

In [13]:
# altair.renderers.enable("browser")

RendererRegistry.enable('browser')

In [36]:
density = (
    alt.Chart(df)
    .transform_filter(interval)
    .transform_fold(
        ["lambda_sec", "lambda_eff", "lambda_kin", "lambda_div"],
        as_=["lambda_type", "value"]
    )
    .transform_calculate(log_value="log(datum.value) / log(10)")
    .mark_bar(opacity=0.4, binSpacing=0)
    .encode(
        x=alt.X("log_value:Q", title=f"log₁₀(Lambda Value)").bin(maxbins=40, base=10),
        y=alt.Y("count()", title="Count", stack=False),
        color=alt.Color("lambda_type:N", title="Lambda", legend=alt.Legend(orient="none", legendX=550, legendY=300)),
    )
    .properties(title="Lambda Distributions (Selected Points)", width=500, height=300)
)


# Build chart
c2 = (text & density)
combined = (combined_scatter | c2).configure_title(fontSize=14, anchor="middle").configure_view(stroke=None)
combined


alt.HConcatChart(...)

In [73]:
df

lambda_sec,lambda_eff,lambda_kin,lambda_div,obj_total,obj_homeo,obj_kin,obj_eff,obj_sec,obj_div
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.000353,0.000006,0.003282,0.003087,0.069609,0.019302,12.179734,301.9015,24.507251,0.003524
0.000075,0.000071,0.000512,0.00861,0.037559,0.000736,31.263228,271.95076,18.360109,0.00344
0.000521,0.000019,0.008738,0.000448,0.144049,0.01998,12.109638,294.656254,24.05123,0.004107
0.000248,0.000044,0.006113,0.000551,0.112809,0.020199,12.109638,285.32349,24.055077,0.004036
0.000015,0.000008,0.003604,0.000869,0.065901,0.019302,12.179708,300.374657,25.082489,0.003564
0.000894,0.000003,0.000245,0.000239,0.018196,9.3057e-13,42.557321,247.575825,7.9001,0.004273
0.000333,0.000013,0.000858,0.000182,0.035973,5.1435e-12,24.609489,359.523478,30.753139,0.004282
0.000373,0.000001,0.000122,0.000894,0.008492,6.4787e-12,42.558155,247.580988,7.900389,0.003477
0.000018,0.000045,0.000204,0.000284,0.019276,0.000127,36.899754,252.250715,12.949534,0.004074


In [110]:
os.getcwd()

'/Users/heenasaqib/dev/vEcoli'

In [10]:
os.chdir(os.path.expanduser('~/dev/vEcoli/notebooks/Heena notebooks/Metabolism_New Genes'))
import polars as pl
# import datafarme
folder = f'pareto_results_shrunk_1000samples/'
df = pl.read_csv(f'{folder}pareto_results.csv')

from pareto_exploration import plot_pairwise_altair
plot_pairwise_altair(df)

  Saved: notebooks/Heena notebooks/Metabolism_New Genes/pareto_results_temp/pairwise_analysis.html


In [113]:
altair.renderers.

Index,lambda_sec,lambda_eff,lambda_kin,lambda_div,obj_total,obj_homeo,obj_kin,obj_eff,obj_sec,obj_div
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1,0.000353,0.000001,0.004847,0.004555,0.087405,0.019359,12.170808,301.558276,24.463608,0.00352
2,0.000075,0.000008,0.000781,0.001542,0.02381,8.3600e-12,22.198826,422.346993,39.450992,0.003473
3,0.000521,0.000002,0.008053,0.000309,0.130572,0.019678,12.135031,298.46213,24.222629,0.004118
4,0.000248,0.000002,0.002003,0.0031,0.050115,0.011418,15.258807,345.062322,29.954243,0.003456
5,0.000015,0.000018,0.000171,0.000148,0.010841,1.3800e-12,30.417783,290.801272,19.831474,0.00406
